In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
!pip install -q transformers datasets sentencepiece sacrebleu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 9.3 MB/s eta 0:00:00


In [4]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu import sentence_bleu
from tqdm.auto import tqdm

In [5]:
dataset = load_dataset("sanganaka/ramayana-anvaya")
print(dataset)

README.md:   0%|          | 0.00/408 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/401k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16447 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1829 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sloka', 'prose'],
        num_rows: 16447
    })
    test: Dataset({
        features: ['sloka', 'prose'],
        num_rows: 1829
    })
})


In [6]:
test_dataset = dataset["test"]
print("Number of test samples:", len(test_dataset))

Number of test samples: 1829


In [7]:
print(test_dataset.column_names)

['sloka', 'prose']


In [8]:
print(test_dataset[0])

{'sloka': 'धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह', 'prose': 'धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः'}


In [9]:
columns = test_dataset.column_names
print("Available columns:")
for i, col in enumerate(columns):
    print(i, ":", col)

Available columns:
0 : sloka
1 : prose


In [10]:
sample = test_dataset[0]
for key, value in sample.items():
    print("\nCOLUMN:", key)
    print("VALUE:", value)


COLUMN: sloka
VALUE: धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह

COLUMN: prose
VALUE: धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः


In [11]:
SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

In [12]:
SLOKA_COLUMN = "YOUR_SLOKA_COLUMN"
PROSE_COLUMN = "YOUR_PROSE_COLUMN"

In [13]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [14]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="san_Deva"
)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [15]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)
model = model.to(device)
model.eval()
print("NLLB model loaded successfully.")

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB model loaded successfully.


In [16]:
SOURCE_LANG = "san_Deva"
TARGET_LANG = "eng_Latn"
target_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)
print("Source language:", SOURCE_LANG)
print("Target language:", TARGET_LANG)
print("Target token ID:", target_token_id)

Source language: san_Deva
Target language: eng_Latn
Target token ID: 256047


In [17]:
def translate_sanskrit(text, max_length=256):
    """Translate Sanskrit text into English using NLLB."""
    if not text:
        return ""
    text = str(text).strip()
    if not text:
        return ""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=target_token_id,
            max_length=max_length,
            num_beams=4
        )
    translation = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0]
    return translation.strip()

In [18]:
print(test_dataset.column_names)

['sloka', 'prose']


In [19]:
print(test_dataset[0])

{'sloka': 'धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह', 'prose': 'धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः'}


In [20]:
for column in test_dataset.column_names:
    print("=" * 80)
    print("COLUMN:", column)
    print("VALUE:")
    print(test_dataset[0][column])

COLUMN: sloka
VALUE:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह
COLUMN: prose
VALUE:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः


In [21]:
print(test_dataset.column_names)
print(test_dataset[0])

['sloka', 'prose']
{'sloka': 'धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह', 'prose': 'धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः'}


In [22]:
SLOKA_COLUMN = "actual_sloka_column"
PROSE_COLUMN = "actual_prose_column"

In [23]:
def translate_batch(texts, batch_size=8, max_length=256):
    translations = []


    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Translating"
    ):
        batch = texts[start:start + batch_size]

        clean_batch = [
            "" if x is None else str(x)
            for x in batch
        ]

        inputs = tokenizer(
            clean_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            translated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=target_token_id,
                max_length=max_length,
                num_beams=4
            )

        batch_translations = tokenizer.batch_decode(
            translated_tokens,
            skip_special_tokens=True
        )

        translations.extend(
            [text.strip() for text in batch_translations]
        )

    return translations

In [24]:
for column in test_dataset.column_names:
    print("=" * 80)
    print("COLUMN:", column)
    print("FIRST VALUE:")
    print(test_dataset[0][column])

COLUMN: sloka
FIRST VALUE:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह
COLUMN: prose
FIRST VALUE:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः


In [25]:
SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

In [26]:
slokas = test_dataset[SLOKA_COLUMN]
proses = test_dataset[PROSE_COLUMN]

print("Number of test samples:", len(test_dataset))
print("Number of slokas:", len(slokas))
print("Number of prose:", len(proses))

Number of test samples: 1829
Number of slokas: 1829
Number of prose: 1829


In [27]:
for i in range(3):
    print("=" * 80)
    print("SAMPLE", i + 1)

    print("\nSLOKA:")
    print(slokas[i])

    print("\nPROSE:")
    print(proses[i])

SAMPLE 1

SLOKA:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह

PROSE:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः
SAMPLE 2

SLOKA:
शबला सा रुदन्ती च क्रोशन्ती चेदमब्रवीत्वसिष्ठस्याग्रतस्स्थित्वा मेघदुन्दुभिराविणी

PROSE:
सा शबला रुदन्ती च क्रोशन्ती च वसिष्ठस्य अग्रत: स्थित्वा मेघदुन्दुभिराविणी इदम् अब्रवीत्
SAMPLE 3

SLOKA:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभासीता नाम वरारोहा वैदेही तनुमध्यमा

PROSE:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभा वैदेही तनुमध्यमा सीता नाम वरारोहा


In [28]:
print("Starting Sloka translation...")

english_slokas = translate_batch(
    slokas,
    batch_size=8,
    max_length=256
)

print("\nSloka translation completed!")
print("Number of translations:", len(english_slokas))

Starting Sloka translation...


Translating:   0%|          | 0/229 [00:00<?, ?it/s]


Sloka translation completed!
Number of translations: 1829


In [29]:
for i in range(3):
    print("=" * 80)
    print("SAMPLE", i + 1)
    print("\nSanskrit Sloka:")
    print(slokas[i])
    print("\nEnglish Sloka:")
    print(english_slokas[i])

SAMPLE 1

Sanskrit Sloka:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह

English Sloka:
Dharmaputrabolvansuheshan is widely believed to have been involved in the battle of the half-blood, including the monkeys and the monkeys, and the monsters and the monkeys, as well as the conflicts with Zeus and the warriors and the multiples.
SAMPLE 2

Sanskrit Sloka:
शबला सा रुदन्ती च क्रोशन्ती चेदमब्रवीत्वसिष्ठस्याग्रतस्स्थित्वा मेघदुन्दुभिराविणी

English Sloka:
Shabla Sa Rudanti and Kroshanti Chadambrviṭavya Sishtasyagṛta is located in Meghdundunduvirāviṇi.
SAMPLE 3

Sanskrit Sloka:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभासीता नाम वरारोहा वैदेही तनुमध्यमा

English Sloka:
Taptakachanchanvarnabha bloodthungankhi शुभासीता name Vararoha Vedhi Tanu intermediama


In [30]:
print("Starting Prose translation...")
english_proses = translate_batch(
    proses,
    batch_size=8,
    max_length=256
)
print("\nProse translation completed!")
print("Number of translations:", len(english_proses))

Starting Prose translation...


Translating:   0%|          | 0/229 [00:00<?, ?it/s]


Prose translation completed!
Number of translations: 1829


In [31]:
for i in range(3):
    print("=" * 80)
    print("SAMPLE", i + 1)

    print("\nSanskrit Prose:")
    print(proses[i])

    print("\nEnglish Prose:")
    print(english_proses[i])

SAMPLE 1

Sanskrit Prose:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः

English Prose:
The epic epic epic of warfare, known as the Dharmasutrao Balwansūshen, is the epic of warfare with horses, monsters, and many others, including warfare with monsters.
SAMPLE 2

Sanskrit Prose:
सा शबला रुदन्ती च क्रोशन्ती च वसिष्ठस्य अग्रत: स्थित्वा मेघदुन्दुभिराविणी इदम् अब्रवीत्

English Prose:
Shabla Ruddanthi and Roshanthi are at the forefront of Vashishtha: the status of Meghdundunduviravini is abrviit.
SAMPLE 3

Sanskrit Prose:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभा वैदेही तनुमध्यमा सीता नाम वरारोहा

English Prose:
Taptakachanchannavarnabha bloodthungankhi shuba vedhi tanumadhyamama sita name vararoha


In [32]:
from sacrebleu import sentence_bleu

def calculate_bleu(candidate, reference):

    if not candidate or not reference:
        return 0.0

    score = sentence_bleu(
        candidate,
        [reference]
    )

    return score.score

In [33]:
bleu_scores = []
for sloka_en, prose_en in tqdm(
    zip(english_slokas, english_proses),
    total=len(english_slokas),
    desc="Calculating BLEU"
):
    score = calculate_bleu(
        sloka_en,
        prose_en
    )

    bleu_scores.append(score)
print("BLEU calculation completed!")
print("Number of BLEU scores:", len(bleu_scores))

Calculating BLEU:   0%|          | 0/1829 [00:00<?, ?it/s]

BLEU calculation completed!
Number of BLEU scores: 1829


In [34]:
for i in range(10):
    print(
        f"Sample {i+1}: BLEU = {bleu_scores[i]:.4f}"
    )

Sample 1: BLEU = 3.5818
Sample 2: BLEU = 2.9048
Sample 3: BLEU = 6.5673
Sample 4: BLEU = 6.6016
Sample 5: BLEU = 11.5288
Sample 6: BLEU = 0.0000
Sample 7: BLEU = 8.8936
Sample 8: BLEU = 2.6342
Sample 9: BLEU = 5.1367
Sample 10: BLEU = 1.5733


In [35]:
results_df = pd.DataFrame({
    "sample_id": range(1, len(test_dataset) + 1),

    "sanskrit_sloka": slokas,

    "sanskrit_prose": proses,

    "english_sloka": english_slokas,

    "english_prose": english_proses,

    "bleu_score": bleu_scores
})

print("Rows:", len(results_df))
print("Columns:", results_df.columns.tolist())

Rows: 1829
Columns: ['sample_id', 'sanskrit_sloka', 'sanskrit_prose', 'english_sloka', 'english_prose', 'bleu_score']


In [36]:
print("Missing values:")
print(results_df.isnull().sum())

Missing values:
sample_id         0
sanskrit_sloka    0
sanskrit_prose    0
english_sloka     0
english_prose     0
bleu_score        0
dtype: int64


In [37]:
print(
    "Empty English Sloka translations:",
    (results_df["english_sloka"].str.strip() == "").sum()
)

print(
    "Empty English Prose translations:",
    (results_df["english_prose"].str.strip() == "").sum()
)

Empty English Sloka translations: 0
Empty English Prose translations: 0


In [38]:
print("BLEU Statistics")
print("=" * 50)

print("Number of samples:", len(results_df))
print("Average BLEU:", results_df["bleu_score"].mean())
print("Minimum BLEU:", results_df["bleu_score"].min())
print("Maximum BLEU:", results_df["bleu_score"].max())
print("Median BLEU:", results_df["bleu_score"].median())

BLEU Statistics
Number of samples: 1829
Average BLEU: 4.560312270375985
Minimum BLEU: 0.0
Maximum BLEU: 79.09371125096185
Median BLEU: 2.971717045858581


In [39]:
results_df.sort_values(
    by="bleu_score",
    ascending=False
)[
    [
        "sample_id",
        "english_sloka",
        "english_prose",
        "bleu_score"
    ]
].head(10)

,sample_id,english_sloka,english_prose,bleu_score
989,990,Selfishness and beliefs and desolation and sel...,Simplicity and beliefs and desolation and maje...,79.093711
1373,1374,Truth and religion and power and ghostly tremb...,Truth and religion and power and ghostly tremb...,51.003234
394,395,I am a male tiger and a teacher of hospitality.,I am a male tiger who is a master of hospitality.,44.127399
1521,1522,"There is no such thing as a pious act, no such...",There is no such thing as Peter's goodness in ...,41.794640
1014,1015,There is a lot of water and a lot of rocks and...,"There is a lot of water and a lot of stones, a...",40.889897
1192,1193,It is possible to make a room outdoors with a ...,It is possible to decorate it with a row of cl...,37.517623
982,983,"Where is the nature of wealth, the princess an...",The nature of the wealth and the quality of th...,35.814692
1095,1096,They turned it over and turned it over to him ...,"They turned it over to him, stopped and asked,...",32.267897
644,645,Artificial weapons or weapons can be used by m...,Artificial weapons or artificial weapons or th...,31.916291
431,432,"And there will be no evil, no unkindness, and ...",Then there will be something unpleasant to see...,31.642572


In [41]:
results_df.sort_values(
    by="bleu_score",
    ascending=True
)[
    [
        "sample_id",
        "english_sloka",
        "english_prose",
        "bleu_score"
    ]
].head(10)

,sample_id,english_sloka,english_prose,bleu_score
1810,1811,"Today is the day of death, the hour of death, ...",Then he passed by: Parmayatt: Musleen Nilum La...,0.0
1820,1821,He grew up in a tree of great beauty and great...,Kuपित: अंगदः सवृक्षम् महाबाहुम् महाबलम् आपतन्त...,0.0
1799,1800,Pramattam Pramattam Pramattam Pramattam Pramat...,"Sharp little gifts are given to the proud, the...",0.0
979,980,"At that time, there was a great deal of sadnes...",Rajen: then: allार्थनाशनीम्: विक्लबाम् बुद्धिम...,0.0
1001,1002,The branches of the branches of the buds are d...,Dharanipatta: Abhishāna: Kadambathasasu Delaye...,0.0
710,711,Vishwanath Vishwanath Vishwanath Vishwanath Vi...,Rakshevar: Ravana: Vibhishnavch: Hearing the w...,0.0
1198,1199,"If you don't come in, Satan will come in and k...",Apgxtat Nश्यध्वम् राघवः सीतम् आप्नोति परममर्षि...,0.0
111,112,"My mother's wombs, my feet, my feet, my feet, ...",Nagottam: Horses and Ghazans and chariots and ...,0.0
768,769,Upgrades are given in the following order: Hap...,Mahabaho अद्य रणात् आनीतम् रामस्य तत् शिरः दृष...,0.0
1157,1158,Satangrityabhyabhyabhyabhyabhyabhyabhyabhyabhy...,Rust: He swallowed aniltuliy velocity with all...,0.0


In [42]:
OUTPUT_FILE = "ramayana_anvaya_nllb_bleu_results.csv"

results_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("CSV created successfully!")
print(OUTPUT_FILE)

CSV created successfully!
ramayana_anvaya_nllb_bleu_results.csv


In [43]:
import os

print("File exists:", os.path.exists(OUTPUT_FILE))
print("File size:", os.path.getsize(OUTPUT_FILE), "bytes")

File exists: True
File size: 1578290 bytes


In [44]:
check_df = pd.read_csv(OUTPUT_FILE)
print("Rows in CSV:", len(check_df))
print("Columns:", check_df.columns.tolist())

Rows in CSV: 1829
Columns: ['sample_id', 'sanskrit_sloka', 'sanskrit_prose', 'english_sloka', 'english_prose', 'bleu_score']


In [47]:
# Check that the output file exists
import os

print("File exists:", os.path.exists(OUTPUT_FILE))
print("File path:", OUTPUT_FILE)

File exists: True
File path: ramayana_anvaya_nllb_bleu_results.csv


In [49]:
import os

print("File exists:", os.path.exists(OUTPUT_FILE))
print("File location:", os.path.abspath(OUTPUT_FILE))

File exists: True
File location: /kaggle/working/ramayana_anvaya_nllb_bleu_results.csv


In [50]:
!ls -lh /kaggle/working/

total 1.6M
-rw-r--r-- 1 root root 1.6M Sep 15 03:06 ramayana_anvaya_nllb_bleu_results.csv


In [51]:
import os

print(os.path.exists("/kaggle/working/ramayana_anvaya_nllb_bleu_results.csv"))

True


In [52]:
!ls -lh /kaggle/working/

total 1.6M
-rw-r--r-- 1 root root 1.6M Sep 15 03:06 ramayana_anvaya_nllb_bleu_results.csv


Task 2 completed here from here TASK 3 started

In [53]:
import os
import pandas as pd
import numpy as np

TASK3_DIR = "/kaggle/working/task3_ramayana"
os.makedirs(TASK3_DIR, exist_ok=True)

TRANSLATION_FILE = os.path.join(
    TASK3_DIR,
    "task3_train_translations.csv"
)

FINAL_FILE = os.path.join(
    TASK3_DIR,
    "ramayana_anvaya_task3_train_results.csv"
)

print("Task-3 folder:", TASK3_DIR)
print("Translation file:", TRANSLATION_FILE)
print("Final file:", FINAL_FILE)

Task-3 folder: /kaggle/working/task3_ramayana
Translation file: /kaggle/working/task3_ramayana/task3_train_translations.csv
Final file: /kaggle/working/task3_ramayana/ramayana_anvaya_task3_train_results.csv


In [56]:
TASK3_DIR = "/content/drive/MyDrive/Ramayana_Task3"

In [57]:
import os

print("Searching /kaggle/working for CSV files...\n")

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

Searching /kaggle/working for CSV files...

/kaggle/working/ramayana_anvaya_nllb_bleu_results.csv


In [58]:
print("train_df exists:", "train_df" in globals())

if "train_df" in globals():
    print("Shape:", train_df.shape)
    print("Columns:")
    print(train_df.columns.tolist())

    print("\nEmpty English Sloka translations:",
          (train_df["english_sloka"].fillna("").astype(str).str.strip() == "").sum())

    print("Empty English Prose translations:",
          (train_df["english_prose"].fillna("").astype(str).str.strip() == "").sum())

train_df exists: False


In [59]:
import os
import pandas as pd

# Kaggle folder
TASK3_DIR = "/kaggle/working/task3_ramayana"
os.makedirs(TASK3_DIR, exist_ok=True)

TRANSLATION_FILE = os.path.join(
    TASK3_DIR,
    "task3_train_translations.csv"
)

print("Task-3 folder:", TASK3_DIR)
print("Translation file:", TRANSLATION_FILE)

print("\nChecking folder:")
print(os.listdir(TASK3_DIR))

Task-3 folder: /kaggle/working/task3_ramayana
Translation file: /kaggle/working/task3_ramayana/task3_train_translations.csv

Checking folder:
[]


In [60]:
# Reload dataset
from datasets import load_dataset

dataset = load_dataset("sanganaka/ramayana-anvaya")

train_dataset = dataset["train"]

print("Training samples:", len(train_dataset))
print("Columns:", train_dataset.column_names)
print(train_dataset[0])

Training samples: 16447
Columns: ['sloka', 'prose']
{'sloka': 'कृत्वा निश्शब्दमेकाग्रा श्श्रुण्वन्तु हरयो ममतत्वं सङ्कीर्तयिष्यामि यथा जानामि मैथिलीम्', 'prose': 'हरयः मैथिलीम् यथा जानामि तत्वम् सङ्कीर्तयिष्यामि निश्शब्दम् कृत्वा एकाग्राः मम श्रुण्वन्तु'}


In [61]:
SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

train_df = pd.DataFrame({
    "sample_id": range(1, len(train_dataset) + 1),
    "sanskrit_sloka": train_dataset[SLOKA_COLUMN],
    "sanskrit_prose": train_dataset[PROSE_COLUMN]
})

train_df["english_sloka"] = ""
train_df["english_prose"] = ""

print("Rows:", len(train_df))
print(train_df.head())

Rows: 16447
   sample_id                                     sanskrit_sloka  \
0          1  कृत्वा निश्शब्दमेकाग्रा श्श्रुण्वन्तु हरयो ममत...   
1          2  कामं वा स्वयमेवाद्य तत्र मां नेतुमर्हसियत्रासौ...   
2          3  अथतान्सचिवांस्तत्रसर्वानाभाष्यरावणः सभांसन्नाद...   
3          4  तं मत्तमातङ्गविलासगामी गच्छन्तमव्यग्रमना महात्...   
4          5  इतीव देवी बहुधा विलप्य सर्वात्मना राममनुस्मरन्...   

                                      sanskrit_prose english_sloka  \
0  हरयः मैथिलीम् यथा जानामि तत्वम् सङ्कीर्तयिष्या...                 
1  वा पुरुषव्याघ्रः मे पुत्रो असौ यत्र तपः तप्यते...                 
2  अथ महाबलः जगत्सन्तापनः क्रूरः राक्षसेश्वरः राव...                 
3  मत्तमातङ्गविलासगामी महात्मा सः लक्ष्मणः गच्छन्...                 
4  देवी इतीव बहुधा विलप्य सर्वात्मना रामम् अनुस्म...                 

  english_prose  
0                
1                
2                
3                
4                


In [62]:
print("Model exists:", "model" in globals())
print("Tokenizer exists:", "tokenizer" in globals())
print("Device:", device if "device" in globals() else "Not defined")

Model exists: True
Tokenizer exists: True
Device: cuda


In [63]:
import os
import pandas as pd

TASK3_DIR = "/kaggle/working/task3_ramayana"
os.makedirs(TASK3_DIR, exist_ok=True)

TRANSLATION_FILE = os.path.join(
    TASK3_DIR,
    "task3_train_translations.csv"
)

SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

train_dataset = dataset["train"]

train_df = pd.DataFrame({
    "sample_id": range(1, len(train_dataset) + 1),
    "sanskrit_sloka": train_dataset[SLOKA_COLUMN],
    "sanskrit_prose": train_dataset[PROSE_COLUMN]
})

# Checkpoint exists?
if os.path.exists(TRANSLATION_FILE):
    print("Checkpoint found!")
    train_df = pd.read_csv(
        TRANSLATION_FILE,
        encoding="utf-8-sig"
    )
else:
    train_df["english_sloka"] = ""
    train_df["english_prose"] = ""
    train_df.to_csv(
        TRANSLATION_FILE,
        index=False,
        encoding="utf-8-sig"
    )
    print("New checkpoint created.")

print("\nRows:", len(train_df))
print("Columns:", train_df.columns.tolist())

New checkpoint created.

Rows: 16447
Columns: ['sample_id', 'sanskrit_sloka', 'sanskrit_prose', 'english_sloka', 'english_prose']


In [64]:
train_df["english_sloka"] = (
    train_df["english_sloka"]
    .fillna("")
    .astype(str)
)

train_df["english_prose"] = (
    train_df["english_prose"]
    .fillna("")
    .astype(str)
)

completed = (
    (train_df["english_sloka"].str.strip() != "") &
    (train_df["english_prose"].str.strip() != "")
)

print("Total samples:", len(train_df))
print("Completed:", completed.sum())
print("Remaining:", (~completed).sum())


Total samples: 16447
Completed: 0
Remaining: 16447


In [67]:
def translate_batch_fast(
    texts,
    batch_size=16,
    max_length=256
):
    translations = []

    tokenizer.src_lang = SOURCE_LANG

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Translating"
    ):

        batch = texts[start:start + batch_size]

        batch = [
            "" if x is None else str(x).strip()
            for x in batch
        ]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():

            output = model.generate(
                **inputs,
                forced_bos_token_id=target_token_id,
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(
            output,
            skip_special_tokens=True
        )

        translations.extend(
            [text.strip() for text in decoded]
        )

        del inputs
        del output

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return translations

print("✅ translate_batch_fast() is ready!")

✅ translate_batch_fast() is ready!


In [68]:
print("Function exists:", "translate_batch_fast" in globals())
print("GPU:", torch.cuda.get_device_name(0))


Function exists: True
GPU: Tesla T4


In [69]:
CHECKPOINT_SIZE = 250
BATCH_SIZE = 16

In [70]:
print("=" * 70)
print("CHECKING TRAINING TRANSLATIONS")
print("=" * 70)

print("Total samples:", len(train_df))

empty_e1 = (
    train_df["english_sloka"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_e2 = (
    train_df["english_prose"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Empty E1 translations:", empty_e1)
print("Empty E2 translations:", empty_e2)

if empty_e1 == 0 and empty_e2 == 0:
    print("\n✅ ALL TRANSLATIONS ARE PRESENT")
else:
    print("\n⚠️ Some translations are missing.")

CHECKING TRAINING TRANSLATIONS
Total samples: 16447
Empty E1 translations: 16447
Empty E2 translations: 16447

⚠️ Some translations are missing.


In [71]:
print("translate_batch_fast exists:",
      "translate_batch_fast" in globals())


translate_batch_fast exists: True


In [72]:
CHECKPOINT_SIZE = 250
BATCH_SIZE = 16

total_samples = len(train_df)

for start in range(0, total_samples, CHECKPOINT_SIZE):

    end = min(start + CHECKPOINT_SIZE, total_samples)

    # Check whether this block is already completed
    block_completed = (
        (train_df.loc[start:end-1, "english_sloka"].str.strip() != "") &
        (train_df.loc[start:end-1, "english_prose"].str.strip() != "")
    )

    if block_completed.all():
        print(f"Skipping completed block: {start} - {end}")
        continue

    print("\n" + "=" * 70)
    print(f"Processing samples {start} - {end}")
    print("=" * 70)

    # ------------------------------------------------
    # SLOKA → E1
    # ------------------------------------------------
    sloka_texts = train_df.loc[
        start:end-1,
        "sanskrit_sloka"
    ].tolist()

    print("Translating Sloka → E1...")

    E1 = translate_batch_fast(
        sloka_texts,
        batch_size=BATCH_SIZE,
        max_length=256
    )

    # ------------------------------------------------
    # PROSE → E2
    # ------------------------------------------------
    prose_texts = train_df.loc[
        start:end-1,
        "sanskrit_prose"
    ].tolist()

    print("Translating Prose → E2...")

    E2 = translate_batch_fast(
        prose_texts,
        batch_size=BATCH_SIZE,
        max_length=256
    )

    # ------------------------------------------------
    # SAVE TRANSLATIONS
    # ------------------------------------------------
    train_df.loc[
        start:end-1,
        "english_sloka"
    ] = E1

    train_df.loc[
        start:end-1,
        "english_prose"
    ] = E2

    # ------------------------------------------------
    # CHECKPOINT SAVE
    # ------------------------------------------------
    train_df.to_csv(
        TRANSLATION_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"✅ Checkpoint saved: {start} - {end}")

print("\n🎉 ALL TRAINING TRANSLATIONS COMPLETED.")


Processing samples 0 - 250
Translating Sloka → E1...


Translating:   0%|          | 0/16 [00:00<?, ?it/s]

Translating Prose → E2...


Translating:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Checkpoint saved: 0 - 250

Processing samples 250 - 500
Translating Sloka → E1...


...

✅ Checkpoint saved: 16000 - 16250

Processing samples 16250 - 16447
Translating Sloka → E1...


Translating:   0%|          | 0/13 [00:00<?, ?it/s]

Translating Prose → E2...


Translating:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Checkpoint saved: 16250 - 16447

🎉 ALL TRAINING TRANSLATIONS COMPLETED.


In [73]:
print("Total samples:", len(train_df))

print(
    "Empty E1:",
    train_df["english_sloka"].fillna("").str.strip().eq("").sum()
)

print(
    "Empty E2:",
    train_df["english_prose"].fillna("").str.strip().eq("").sum()
)

Total samples: 16447
Empty E1: 0
Empty E2: 0


In [74]:
!pip install -q sacrebleu bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00


In [75]:
from sacrebleu import sentence_chrf
from bert_score import score
from tqdm.auto import tqdm

print("✅ Metric libraries loaded")

✅ Metric libraries loaded


In [76]:
def calculate_chrf(candidate, reference):
    if not candidate or not reference:
        return 0.0

    result = sentence_chrf(
        candidate,
        [reference]
    )

    return result.score

In [77]:
print("=" * 70)
print("CALCULATING CHRF")
print("=" * 70)

chrf_sloka_e1 = []
chrf_prose_e2 = []
chrf_e1_e2 = []

for _, row in tqdm(
    train_df.iterrows(),
    total=len(train_df),
    desc="Calculating CHRF"
):

    sloka = str(row["sanskrit_sloka"])
    prose = str(row["sanskrit_prose"])
    e1 = str(row["english_sloka"])
    e2 = str(row["english_prose"])

    # 1. Sloka vs English Sloka (E1)
    score_1 = calculate_chrf(
        sloka,
        e1
    )

    # 2. Prose vs English Prose (E2)
    score_2 = calculate_chrf(
        prose,
        e2
    )

    # 3. English Sloka vs English Prose
    score_3 = calculate_chrf(
        e1,
        e2
    )

    chrf_sloka_e1.append(score_1)
    chrf_prose_e2.append(score_2)
    chrf_e1_e2.append(score_3)

print("\n✅ CHRF calculation completed!")

CALCULATING CHRF


Calculating CHRF:   0%|          | 0/16447 [00:00<?, ?it/s]


✅ CHRF calculation completed!


In [78]:
train_df["chrf_sloka_e1"] = chrf_sloka_e1
train_df["chrf_prose_e2"] = chrf_prose_e2
train_df["chrf_e1_e2"] = chrf_e1_e2

print("✅ CHRF columns added")

display(
    train_df[
        [
            "sample_id",
            "chrf_sloka_e1",
            "chrf_prose_e2",
            "chrf_e1_e2"
        ]
    ].head(10)
)

✅ CHRF columns added


,sample_id,chrf_sloka_e1,chrf_prose_e2,chrf_e1_e2
0,1,0.000000,0.000000,24.410950
1,2,0.000000,0.000000,7.082218
2,3,3.602832,0.000000,12.546120
3,4,0.000000,0.000000,57.791849
4,5,0.000000,0.000000,23.221143
5,6,0.000000,0.000000,32.209057
6,7,0.350877,0.424088,43.386831
7,8,0.000000,0.000000,12.999641
8,9,0.000000,0.000000,19.470462
9,10,0.000000,0.000000,27.236292


In [79]:
print("=" * 70)
print("CALCULATING BERTSCORE")
print("=" * 70)

E1 = (
    train_df["english_sloka"]
    .fillna("")
    .astype(str)
    .tolist()
)

E2 = (
    train_df["english_prose"]
    .fillna("")
    .astype(str)
    .tolist()
)

print("E1 samples:", len(E1))
print("E2 samples:", len(E2))

P, R, F1 = score(
    cands=E1,
    refs=E2,
    lang="en",
    model_type="roberta-large",
    batch_size=16,
    verbose=True
)

print("\n✅ BERTScore calculation completed!")

CALCULATING BERTSCORE
E1 samples: 16447
E2 samples: 16447


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/2054 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1028 [00:00<?, ?it/s]

done in 424.36 seconds, 38.76 sentences/sec

✅ BERTScore calculation completed!


In [80]:
train_df["bertscore_precision"] = P.cpu().numpy()
train_df["bertscore_recall"] = R.cpu().numpy()
train_df["bertscore_f1"] = F1.cpu().numpy()

print("✅ BERTScore columns added")

display(
    train_df[
        [
            "sample_id",
            "bertscore_precision",
            "bertscore_recall",
            "bertscore_f1"
        ]
    ].head(10)
)

✅ BERTScore columns added


,sample_id,bertscore_precision,bertscore_recall,bertscore_f1
0,1,0.902274,0.902243,0.902258
1,2,0.690850,0.816714,0.748528
2,3,0.751223,0.828902,0.788153
3,4,0.895163,0.897255,0.896208
4,5,0.809343,0.848937,0.828667
5,6,0.855487,0.853333,0.854408
6,7,0.872030,0.888564,0.880220
7,8,0.800863,0.856554,0.827773
8,9,0.847723,0.850639,0.849178
9,10,0.870164,0.821252,0.845001


In [81]:
print("=" * 70)
print("FINAL TASK-3 COLUMNS")
print("=" * 70)

print("Rows:", len(train_df))
print("Columns:", len(train_df.columns))

for i, column in enumerate(train_df.columns, 1):
    print(i, column)

FINAL TASK-3 COLUMNS
Rows: 16447
Columns: 11
1 sample_id
2 sanskrit_sloka
3 sanskrit_prose
4 english_sloka
5 english_prose
6 chrf_sloka_e1
7 chrf_prose_e2
8 chrf_e1_e2
9 bertscore_precision
10 bertscore_recall
11 bertscore_f1


In [82]:
print("=" * 70)
print("QUALITY CHECK")
print("=" * 70)

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nEmpty E1:",
      train_df["english_sloka"].str.strip().eq("").sum())

print("Empty E2:",
      train_df["english_prose"].str.strip().eq("").sum())

print("Missing CHRF Sloka-E1:",
      train_df["chrf_sloka_e1"].isna().sum())

print("Missing CHRF Prose-E2:",
      train_df["chrf_prose_e2"].isna().sum())

print("Missing CHRF E1-E2:",
      train_df["chrf_e1_e2"].isna().sum())

print("Missing BERTScore F1:",
      train_df["bertscore_f1"].isna().sum())

QUALITY CHECK

Missing values:
sample_id              0
sanskrit_sloka         0
sanskrit_prose         0
english_sloka          0
english_prose          0
chrf_sloka_e1          0
chrf_prose_e2          0
chrf_e1_e2             0
bertscore_precision    0
bertscore_recall       0
bertscore_f1           0
dtype: int64

Empty E1: 0
Empty E2: 0
Missing CHRF Sloka-E1: 0
Missing CHRF Prose-E2: 0
Missing CHRF E1-E2: 0
Missing BERTScore F1: 0


In [83]:
print("=" * 70)
print("TASK-3 METRIC SUMMARY")
print("=" * 70)

print("\nCHRF Sloka vs E1:")
print("Mean:", train_df["chrf_sloka_e1"].mean())

print("\nCHRF Prose vs E2:")
print("Mean:", train_df["chrf_prose_e2"].mean())

print("\nCHRF E1 vs E2:")
print("Mean:", train_df["chrf_e1_e2"].mean())

print("\nBERTScore Precision:")
print("Mean:", train_df["bertscore_precision"].mean())

print("\nBERTScore Recall:")
print("Mean:", train_df["bertscore_recall"].mean())

print("\nBERTScore F1:")
print("Mean:", train_df["bertscore_f1"].mean())

TASK-3 METRIC SUMMARY

CHRF Sloka vs E1:
Mean: 1.5702540563110228

CHRF Prose vs E2:
Mean: 3.092373371452187

CHRF E1 vs E2:
Mean: 25.313554383340616

BERTScore Precision:
Mean: 0.8427535

BERTScore Recall:
Mean: 0.8393805

BERTScore F1:
Mean: 0.84016573


In [84]:
FINAL_FILE = "/kaggle/working/task3_ramayana/ramayana_anvaya_task3_train_results.csv"

train_df.to_csv(
    FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("🎉 TASK-3 FINAL CSV CREATED")
print("=" * 70)

print("File:", FINAL_FILE)
print("Rows:", len(train_df))
print("Columns:", len(train_df.columns))

🎉 TASK-3 FINAL CSV CREATED
File: /kaggle/working/task3_ramayana/ramayana_anvaya_task3_train_results.csv
Rows: 16447
Columns: 11


In [85]:
import os

print("File exists:", os.path.exists(FINAL_FILE))

if os.path.exists(FINAL_FILE):
    size_mb = os.path.getsize(FINAL_FILE) / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")

File exists: True
File size: 13.26 MB


In [86]:
final_check = pd.read_csv(
    FINAL_FILE,
    encoding="utf-8-sig"
)

print("Rows:", len(final_check))
print("Columns:", len(final_check.columns))

print("\nColumns:")
print(final_check.columns.tolist())

display(final_check.head())

Rows: 16447
Columns: 11

Columns:
['sample_id', 'sanskrit_sloka', 'sanskrit_prose', 'english_sloka', 'english_prose', 'chrf_sloka_e1', 'chrf_prose_e2', 'chrf_e1_e2', 'bertscore_precision', 'bertscore_recall', 'bertscore_f1']


,sample_id,sanskrit_sloka,sanskrit_prose,english_sloka,english_prose,chrf_sloka_e1,chrf_prose_e2,chrf_e1_e2,bertscore_precision,bertscore_recall,bertscore_f1
0,1,कृत्वा निश्शब्दमेकाग्रा श्श्रुण्वन्तु हरयो ममत...,हरयः मैथिलीम् यथा जानामि तत्वम् सङ्कीर्तयिष्या...,"Doing it quietly, I listened to my voice, and ...","As I know, I will enclose the elements, silent...",0.000000,0.0,24.410950,0.902274,0.902243,0.902258
1,2,कामं वा स्वयमेवाद्य तत्र मां नेतुमर्हसियत्रासौ...,वा पुरुषव्याघ्रः मे पुत्रो असौ यत्र तपः तप्यते...,"When there is a work or a self-development, th...",Male or Tiger: I have sons and you should take...,0.000000,0.0,7.082218,0.690850,0.816714,0.748528
2,3,अथतान्सचिवांस्तत्रसर्वानाभाष्यरावणः सभांसन्नाद...,अथ महाबलः जगत्सन्तापनः क्रूरः राक्षसेश्वरः राव...,Atatanschivanastatraसर्वानाभाष्यरावण: the asse...,At this Mahabal: the end of the world: cruel: ...,3.602832,0.0,12.546120,0.751223,0.828901,0.788153
3,4,तं मत्तमातङ्गविलासगामी गच्छन्तमव्यग्रमना महात्...,मत्तमातङ्गविलासगामी महात्मा सः लक्ष्मणः गच्छन्...,Then Matmatangavillasagami Gachantamvyagarmana...,Matmatangavillasagami Mahatma He is the Lakshm...,0.000000,0.0,57.791849,0.895163,0.897255,0.896208
4,5,इतीव देवी बहुधा विलप्य सर्वात्मना राममनुस्मरन्...,देवी इतीव बहुधा विलप्य सर्वात्मना रामम् अनुस्म...,This goddess is often worshipped as the Suprem...,The goddess is often wrapped in the soul of th...,0.000000,0.0,23.221143,0.809343,0.848937,0.828667


In [87]:
from IPython.display import FileLink

FileLink(FINAL_FILE)


/kaggle/working/task3_ramayana/ramayana_anvaya_task3_train_results.csv

In [88]:
import os

print(os.listdir("/kaggle/working"))
print(os.listdir("/kaggle/working/task3_ramayana"))


['ramayana_anvaya_nllb_bleu_results.csv', 'task3_ramayana', '.virtual_documents']
['ramayana_anvaya_task3_train_results.csv', 'task3_train_translations.csv']


In [89]:
from IPython.display import FileLink

FileLink(
    "/kaggle/working/task3_ramayana/ramayana_anvaya_task3_train_results.csv"
)

/kaggle/working/task3_ramayana/ramayana_anvaya_task3_train_results.csv

In [90]:
required_columns = [
    "sample_id",
    "sanskrit_sloka",
    "sanskrit_prose",
    "english_sloka",
    "english_prose",
    "chrf_sloka_e1",
    "chrf_prose_e2",
    "chrf_e1_e2",
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1"
]

print("=" * 70)
print("TASK-3 FINAL VERIFICATION")
print("=" * 70)

# 1. Number of samples
print("\n1. TRAINING SAMPLES")
print("Rows:", len(train_df))

# 2. Check columns
print("\n2. REQUIRED COLUMNS")

missing = []

for column in required_columns:
    if column in train_df.columns:
        print("✅", column)
    else:
        print("❌", column)
        missing.append(column)

# 3. Check translations
print("\n3. TRANSLATIONS")

empty_e1 = (
    train_df["english_sloka"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_e2 = (
    train_df["english_prose"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Empty E1:", empty_e1)
print("Empty E2:", empty_e2)

# 4. Check metrics
print("\n4. METRICS")

metric_columns = [
    "chrf_sloka_e1",
    "chrf_prose_e2",
    "chrf_e1_e2",
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1"
]

for column in metric_columns:
    if column in train_df.columns:
        print(
            column,
            "→ missing:",
            train_df[column].isna().sum()
        )

# 5. Final result
print("\n" + "=" * 70)

if (
    len(train_df) == 16447
    and len(missing) == 0
    and empty_e1 == 0
    and empty_e2 == 0
    and all(train_df[col].isna().sum() == 0 for col in metric_columns)
):
    print("🎉 TASK-3 COMPLETED SUCCESSFULLY")
    print("✅ Training CSV")
    print("✅ E1 translations")
    print("✅ E2 translations")
    print("✅ CHRF Sloka-E1")
    print("✅ CHRF Prose-E2")
    print("✅ CHRF E1-E2")
    print("✅ BERTScore Precision")
    print("✅ BERTScore Recall")
    print("✅ BERTScore F1")
else:
    print("⚠️ TASK-3 IS NOT 100% COMPLETE")
    print("Please check the errors above.")

print("=" * 70)

TASK-3 FINAL VERIFICATION

1. TRAINING SAMPLES
Rows: 16447

2. REQUIRED COLUMNS
✅ sample_id
✅ sanskrit_sloka
✅ sanskrit_prose
✅ english_sloka
✅ english_prose
✅ chrf_sloka_e1
✅ chrf_prose_e2
✅ chrf_e1_e2
✅ bertscore_precision
✅ bertscore_recall
✅ bertscore_f1

3. TRANSLATIONS
Empty E1: 0
Empty E2: 0

4. METRICS
chrf_sloka_e1 → missing: 0
chrf_prose_e2 → missing: 0
chrf_e1_e2 → missing: 0
bertscore_precision → missing: 0
bertscore_recall → missing: 0
bertscore_f1 → missing: 0

🎉 TASK-3 COMPLETED SUCCESSFULLY
✅ Training CSV
✅ E1 translations
✅ E2 translations
✅ CHRF Sloka-E1
✅ CHRF Prose-E2
✅ CHRF E1-E2
✅ BERTScore Precision
✅ BERTScore Recall
✅ BERTScore F1
